# AML Data Preprocessing

Preprocesses raw AML transaction data into a formatted CSV schema.


## 1. Load Dataset


In [1]:
import os
import sys
import glob
import pandas as pd
from datetime import datetime

SAFE_NROWS = 5_000_000
kaggle_matches = glob.glob("/kaggle/input/**/HI-Large_Trans.csv", recursive=True)

if kaggle_matches:
    csv_path = kaggle_matches[0]
    df = pd.read_csv(csv_path, nrows=SAFE_NROWS)
else:
    sys.path.insert(0, os.path.dirname(os.path.abspath("get_dataset.py")))
    from get_dataset import load_aml_dataset
    df = load_aml_dataset()

print(f"Loaded {len(df):,} rows")


Loaded 5,000,000 rows
Columns : ['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


## 2. Parse Timestamp


In [2]:
dt_series = pd.to_datetime(df["Timestamp"], format="%Y/%m/%d %H:%M")
first_dt = dt_series.min()
start_time = datetime(first_dt.year, first_dt.month, first_dt.day)
firstTs = start_time.timestamp() - 10
df["Timestamp"] = (dt_series.astype("int64") // 10**9 - firstTs).astype("int64")

range_buffer = df["Timestamp"].max() - df["Timestamp"].min()
print(f"Timestamp range: {range_buffer} seconds")


Reference epoch (firstTs): 1659292190.0
Timestamp range: 5938140 seconds


,Timestamp
0,20830
1,19930
2,20830


## 3. Label-Encode Categorical Columns


In [3]:
def encode_columns_shared(df, cols):
    shared = {}
    result = {}
    for col in cols:
        encoded = []
        for val in df[col]:
            if val not in shared:
                shared[val] = len(shared)
            encoded.append(shared[val])
        result[col] = encoded
    return result, shared

def encode_column(series):
    label_dict = {}
    encoded = []
    for val in series:
        if val not in label_dict:
            label_dict[val] = len(label_dict)
        encoded.append(label_dict[val])
    return encoded, label_dict

currency_encoded, currency_dict = encode_columns_shared(df, ['Receiving Currency', 'Payment Currency'])
df['Receiving Currency'] = currency_encoded['Receiving Currency']
df['Payment Currency'] = currency_encoded['Payment Currency']
fmt_encoded, fmt_dict = encode_column(df['Payment Format'])
df['Payment Format'] = fmt_encoded
print(f"Encoded {len(currency_dict)} currencies and {len(fmt_dict)} payment formats")


Currency encoding :  {'US Dollar': 0, 'Euro': 1, 'UK Pound': 2, 'Bitcoin': 3, 'Yen': 4, 'Yuan': 5, 'Canadian Dollar': 6, 'Rupee': 7, 'Australian Dollar': 8, 'Ruble': 9, 'Shekel': 10, 'Brazil Real': 11, 'Mexican Peso': 12, 'Swiss Franc': 13, 'Saudi Riyal': 14}
Payment Format enc :  {'Reinvestment': 0, 'Cheque': 1, 'Credit Card': 2, 'ACH': 3, 'Wire': 4, 'Cash': 5, 'Bitcoin': 6}


,Receiving Currency,Payment Currency,Payment Format
0,0,0,0
1,0,0,0
2,0,0,0


## 4. Build Account Node IDs


In [4]:
account_dict = {}

def get_account_id(bank, acc):
    key = str(bank) + str(acc)
    if key not in account_dict:
        account_dict[key] = len(account_dict)
    return account_dict[key]

to_acc_col = 'Account.1' if 'Account.1' in df.columns else df.columns[4]
df['from_id'] = [get_account_id(b, a) for b, a in zip(df['From Bank'], df['Account'])]
df['to_id'] = [get_account_id(b, a) for b, a in zip(df['To Bank'], df[to_acc_col])]
print(f"Total unique account nodes: {len(account_dict):,}")


Total unique accounts (nodes) : 1754264


,From Bank,Account,from_id,To Bank,Account.1,to_id
0,20,800104D70,0,20,800104D70,0
1,3196,800107150,1,3196,800107150,1
2,1208,80010E430,2,1208,80010E430,2


## 5. Format Columns and Optimize Data Types


In [5]:
df.insert(0, 'EdgeID', range(len(df)))
df.rename(columns={'Amount Paid': 'Amount Sent', 'Payment Currency': 'Sent Currency', 'Receiving Currency': 'Received Currency'}, inplace=True)
cols_to_drop = ['From Bank', 'Account', 'To Bank', to_acc_col]
df.drop(columns=cols_to_drop, inplace=True)
df = df[['EdgeID', 'from_id', 'to_id', 'Timestamp', 'Amount Sent', 'Sent Currency', 'Amount Received', 'Received Currency', 'Payment Format', 'Is Laundering']]
df['EdgeID'] = df['EdgeID'].astype('int32')
df['from_id'] = df['from_id'].astype('int32')
df['to_id'] = df['to_id'].astype('int32')
df['Timestamp'] = df['Timestamp'].astype('int32')
df['Amount Sent'] = df['Amount Sent'].astype('float32')
df['Sent Currency'] = df['Sent Currency'].astype('int16')
df['Amount Received'] = df['Amount Received'].astype('float32')
df['Received Currency'] = df['Received Currency'].astype('int16')
df['Payment Format'] = df['Payment Format'].astype('int8')
df['Is Laundering'] = df['Is Laundering'].astype('int8')
print(f"Memory usage: {df.memory_usage().sum() / 1e6:.1f} MB")


Memory usage optimized: 150.0 MB
Columns: ['EdgeID', 'from_id', 'to_id', 'Timestamp', 'Amount Sent', 'Sent Currency', 'Amount Received', 'Received Currency', 'Payment Format', 'Is Laundering']


,EdgeID,from_id,to_id,Timestamp,Amount Sent,Sent Currency,Amount Received,Received Currency,Payment Format,Is Laundering
0,0,0,0,20830,6794.629883,0,6794.629883,0,0,0
1,1,1,1,19930,7739.290039,0,7739.290039,0,0,0
2,2,2,2,20830,1880.229980,0,1880.229980,0,0,0


## 6. Sort Chronologically


In [6]:
df.sort_values('Timestamp', inplace=True)
df.reset_index(drop=True, inplace=True)
print(f"Processed shape: {df.shape}")
print(f"Laundering ratio: {df['Is Laundering'].sum():,} / {len(df):,} ({df['Is Laundering'].mean()*100:.3f}%)")


Shape : (5000000, 10)
Timestamp range : 5938140 seconds
Laundering done : 2800
Laundering not done : 4997200
Illicit ratio : 2,800 / 5,000,000 = 0.056%


,EdgeID,from_id,to_id,Timestamp,Amount Sent,Sent Currency,Amount Received,Received Currency,Payment Format,Is Laundering
0,1253484,920505,935525,19810,2825.699951,10,2825.699951,10,3,0
1,1284010,958125,1151467,19810,272.869995,14,272.869995,14,2,0
2,370223,276846,276846,19810,724.080017,0,724.080017,0,0,0
3,40731,30371,30371,19810,9362.110352,0,9362.110352,0,0,0
4,40733,30371,30371,19810,5.780000,0,5.780000,0,0,0


## 7. Save Formatted CSV


In [7]:
save_path = "formatted_transactions.csv"
df.to_csv(save_path, index=False)
print(f"Saved formatted_transactions.csv -> {save_path} ({os.path.getsize(save_path) / 1e6:.2f} MB)")


Saved formatted_transactions.csv -> /home/shreyas-nalle/Desktop/Delusional/model/formatted_transactions.csv
File size: 260.51 MB
